# 第 17 课：PGS 动态修正——apd、rpl、rg 与实时字幕

本课把流式 API 返回的碎片恢复成用户看到的文本。这里的 PGS 指动态修正结果协议，而不是声学模型。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 流式 ASR |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 16 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | PGS apd、PGS rpl/rg、片段状态 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：PGS apd、PGS rpl/rg、片段状态。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：CTC collapse 与 prefix 状态；因果/非因果上下文；整段结果的基线。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](16_流式编码器_因果卷积与ChunkAttention.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：按时间到达的 chunk、前端/模型/解码器状态
  ↓ 本课要学会的变换、状态或判断
输出：可与离线对照的 partial/final、状态更新和延迟指标
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

from ipywidgets import interact, IntSlider

项目根目录: <REPO_ROOT>


## 1. 三个字段

- `pgs="apd"`：追加本片结果；
- `pgs="rpl"`：替换历史片段；
- `rg=[a,b]`：替换返回序号区间（边界语义必须以具体 API 文档为准）。

识别结果会修正，是因为新音频和语言上下文改变了之前的判断。

In [2]:
events=[
 {"sn":1,"pgs":"apd","text":"今天"},
 {"sn":2,"pgs":"apd","text":"天气"},
 {"sn":3,"pgs":"apd","text":"真"},
 {"sn":4,"pgs":"apd","text":"热"},
 {"sn":5,"pgs":"rpl","rg":[3,4],"text":"真不错"},
 {"sn":6,"pgs":"apd","text":"。"},
]

def apply_pgs(events):
    slices={}
    snapshots=[]
    for e in events:
        if e["pgs"]=="rpl":
            a,b=e["rg"]
            for sn in range(a,b+1): slices.pop(sn,None)
        slices[e["sn"]]=e["text"]
        snapshots.append("".join(slices[k] for k in sorted(slices)))
    return snapshots

for e,text in zip(events,apply_pgs(events)): print(e,"=>",text)

{'sn': 1, 'pgs': 'apd', 'text': '今天'} => 今天
{'sn': 2, 'pgs': 'apd', 'text': '天气'} => 今天天气
{'sn': 3, 'pgs': 'apd', 'text': '真'} => 今天天气真
{'sn': 4, 'pgs': 'apd', 'text': '热'} => 今天天气真热
{'sn': 5, 'pgs': 'rpl', 'rg': [3, 4], 'text': '真不错'} => 今天天气真不错
{'sn': 6, 'pgs': 'apd', 'text': '。'} => 今天天气真不错。


## 2. 交互播放服务端返回

In [3]:
snapshots=apply_pgs(events)
@interact(step=IntSlider(min=1,max=len(events),value=1,description="返回序号"))
def replay(step=1):
    for i in range(step): print(f"event {i+1}:",events[i])
    print("\n用户界面显示:",snapshots[step-1])

interactive(children=(IntSlider(value=1, description='返回序号', max=6, min=1), Output()), _dom_classes=('widget-i…

## 3. 工业客户端不能只做字符串追加

至少要处理：重复包、乱序包、断线重连、未知 `rg`、最终标记、标点片段以及 UI 光标位置。推荐保存结构化 slice，而不是只保存一个不断拼接的大字符串。

In [4]:
class PGSBuffer:
    def __init__(self): self.parts={}; self.seen=set(); self.final=False
    def accept(self,event):
        sn=event["sn"]
        if sn in self.seen: return self.text()
        self.seen.add(sn)
        if event.get("pgs")=="rpl":
            a,b=event["rg"]
            for old in range(a,b+1): self.parts.pop(old,None)
        self.parts[sn]=event.get("text","")
        self.final=self.final or event.get("ls",False)
        return self.text()
    def text(self): return "".join(self.parts[k] for k in sorted(self.parts))

b=PGSBuffer()
for e in events: print(b.accept(e))
print("重复发送最后一包:",b.accept(events[-1]))

今天
今天天气
今天天气真
今天天气真热
今天天气真不错
今天天气真不错。
重复发送最后一包: 今天天气真不错。


## 4. PGS 与 CTC 的关系

CTC prefix beam search 产生随时间变化的候选；PGS 是把“追加/替换”变化传给客户端的一种协议。CTC 不强制使用 PGS，PGS 也不限定后端必须是 CTC。

## 本课测试

1. `rpl` 为什么不能当成追加？
2. `rg=[2,5]` 通常表示什么？
3. 客户端为什么要按 `sn` 保存片段？
4. PGS 是否等于 CTC 解码算法？
5. partial 文本是否应该立即写入不可修改的业务记录？

<details><summary>展开参考答案</summary>

1. 它会令旧假设失效。2. 替换第 2～5 次返回结果，仍需服从具体接口定义。3. 便于替换、乱序和去重。4. 不是，它是结果更新协议。5. 通常不应，应等待 stable/final 或建立可修订记录。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 17 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `PGS apd`、`PGS rpl/rg`、`片段状态`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**同一 sn 重复或 rpl 包乱序到达**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**实现幂等 PGSBuffer 并加入非法 rg 测试**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**连接 decoder partial 与客户端显示**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：PGS apd、PGS rpl/rg、片段状态。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 PGS apd、PGS rpl/rg、片段状态。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
